# 6.3 - Controlled Generation

**Module 6 - Image & Multimodal Generation** | Netsetos GenAI Engineering

This notebook works through Controlled Generation hands-on: ControlNet: condition on a structure map; IP-Adapter: an image as a prompt; Composing control: multi-ControlNet; Choosing the right control tool; Control in production: reproducible and consent-gated.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

The first code cell is environment prep. It leaves the heavy install (`torch`) commented out for Colab and only flags that the lesson relies on seeded randomness for reproducibility - the real pipeline installs come inline in each later cell (`diffusers`, `controlnet_aux`, etc.).

In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# Uncomment the next line if needed:
# !pip install torch -q

# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

A minimal setup cell - no imports actually execute here, just a commented pip line and a note about reproducibility. Nothing runs; it exists to mark that the notebook assumes a torch/diffusers environment and seeded values.

**How the code works - step by step**
- **Commented `pip install torch`** - uncomment only in a fresh Colab; per-tool installs (`diffusers`, `controlnet_aux`, `transformers`, `accelerate`) appear as comments at the top of the cells that need them.
- **Reproducibility note** - a reminder that the lesson pins seeds so results are repeatable rather than seed-luck (the idea that pays off in the production cell).

**In one line:** environment prep, no execution.

**What you'll see:** (no output - environment prep)

## 1 - ControlNet: condition on a structure map

ControlNet is how you force generation to follow a spatial layout - a pose, edges, or depth extracted from a source image. The move is three steps: turn the source into a *condition map*, load the ControlNet branch onto the frozen base model, then generate with a `conditioning_scale` dial that sets how strictly the output traces the map.

> **Needs a GPU** - loads SDXL in fp16 onto CUDA; shown for reference.

In [ ]:
# pip install diffusers controlnet_aux transformers accelerate torch
import torch
from diffusers import StableDiffusionXLControlNetPipeline, ControlNetModel
from diffusers.utils import load_image
from controlnet_aux import CannyDetector

# STEP 1: preprocess the source into a CONDITION MAP (not the raw photo!)
canny = CannyDetector()
source = load_image("building.png")
edges = canny(source, low_threshold=100, high_threshold=200)  # edge map; low/high = edge sensitivity (100/200 are common defaults)

# STEP 2: load the ControlNet branch + the base pipeline
controlnet = ControlNetModel.from_pretrained(
    "diffusers/controlnet-canny-sdxl-1.0", torch_dtype=torch.float16)
pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    controlnet=controlnet, torch_dtype=torch.float16).to("cuda")

# STEP 3: generate - the prompt fills in content, the edges fix structure
image = pipe(
    prompt="a neon cyberpunk tower at night, highly detailed",
    image=edges,
    controlnet_conditioning_scale=0.7,  # 0=ignore edges, 1=trace rigidly
    num_inference_steps=28,
).images[0]
image.save("controlled.png")
print("same edges, new content")
# Output: same edges, new content

**Explanation**

This cell is the canonical ControlNet recipe, not a toy - it runs a real SDXL pipeline with a Canny-edge condition. The key idea is that you never pass the raw photo: a preprocessor first extracts a structure map, and only that map conditions the frozen denoiser while the text prompt supplies content.

**How the code works - step by step**
- **STEP 1 - `CannyDetector`** turns `building.png` into an edge map (`edges`) with `low/high_threshold` controlling edge sensitivity. You pass this map as `image=`, never the original photo - the classic beginner bug.
- **STEP 2 - `ControlNetModel.from_pretrained`** loads the Canny ControlNet branch in fp16, and `StableDiffusionXLControlNetPipeline` attaches it to the frozen SDXL base on CUDA.
- **STEP 3 - `pipe(...)`** generates: the prompt fills in content, `image=edges` fixes structure, and `controlnet_conditioning_scale=0.7` sets adherence (0 ignores the edges, 1 traces them rigidly).
- **`num_inference_steps=28`** denoising steps; the result is saved to `controlled.png`.

**In one line:** extract a map -> attach the ControlNet branch -> generate with a scale dial.

**What you'll see:** Prints `same edges, new content` and writes `controlled.png` - a new cyberpunk-tower image that keeps the source building's edge layout.

## 2 - IP-Adapter: an image as a prompt

Where ControlNet pins structure, IP-Adapter transfers a *look* - style, palette, or subject - from a reference image, with no training. You load the adapter once onto a normal SDXL pipeline, then hand it a reference picture that it treats as a second, visual prompt alongside your text.

> **Needs a GPU** - loads SDXL in fp16 onto CUDA; shown for reference.

In [ ]:
import torch
from diffusers import StableDiffusionXLPipeline
from diffusers.utils import load_image

pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0", torch_dtype=torch.float16
).to("cuda")

# load the IP-Adapter once; it adds image cross-attention to the frozen model
pipe.load_ip_adapter("h94/IP-Adapter", subfolder="sdxl_models",
                     weight_name="ip-adapter_sdxl.bin")

reference = load_image("style_reference.jpg")   # the LOOK we want
pipe.set_ip_adapter_scale(0.6)                    # 0=ignore image, 1=copy it hard

image = pipe(
    prompt="a mountain palace at golden hour",       # the CONTENT we want
    ip_adapter_image=reference,                        # the reference as a visual prompt
    num_inference_steps=28,
).images[0]
image.save("styled.png")
print("content from text, look from the reference")
# Output: content from text, look from the reference

**Explanation**

This cell shows IP-Adapter as an add-on to the frozen model: `load_ip_adapter` bolts image cross-attention onto a standard SDXL pipeline, and a reference image becomes a prompt encoded via CLIP. It is the same 'attach, don't retrain' pattern as ControlNet, but conditioning on appearance instead of geometry.

**How the code works - step by step**
- **`StableDiffusionXLPipeline.from_pretrained`** loads the plain SDXL base in fp16 on CUDA (no ControlNet here).
- **`load_ip_adapter`** adds image-conditioning cross-attention from the `h94/IP-Adapter` weights - load once, reuse.
- **`load_image` + `ip_adapter_image=reference`** passes `style_reference.jpg` as a visual prompt; the model encodes it with CLIP and uses those features.
- **`set_ip_adapter_scale(0.6)`** dials strength: ~0.6 blends the reference's look with the prompt, near 1.0 copies it hard, near 0 ignores it.
- **`pipe(...)`** generates with content from the text prompt and look from the reference, saving `styled.png`.

**In one line:** load the adapter once -> pass a reference image -> the model uses its CLIP features as a visual prompt.

**What you'll see:** Prints `content from text, look from the reference` and writes `styled.png` - a golden-hour mountain palace rendered in the reference image's style.

## 3 - Composing control: multi-ControlNet

One control rarely captures a full brief. Here two ControlNets stack - a pose skeleton fixes the body, a depth map fixes the 3D layout - each with its own preprocessed map and its own scale. The same pattern extends to union models (one model, many condition types) and regional prompting (a prompt per area).

> **Needs a GPU** - loads SDXL plus two ControlNets in fp16 onto CUDA; shown for reference.

In [ ]:
import torch
from diffusers import StableDiffusionXLControlNetPipeline, ControlNetModel

# MULTI-CONTROLNET: pass a LIST of ControlNets, each with its own scale.
# Here: a pose skeleton fixes the body, a depth map fixes the 3D layout.
pose_cn  = ControlNetModel.from_pretrained("thibaud/controlnet-openpose-sdxl-1.0",
                                           torch_dtype=torch.float16)
depth_cn = ControlNetModel.from_pretrained("diffusers/controlnet-depth-sdxl-1.0",
                                           torch_dtype=torch.float16)

# Each ControlNet still needs its OWN preprocessed map (exactly like Step 2)
# - never the raw photo. Build one condition map per ControlNet:
from controlnet_aux import OpenposeDetector, MidasDetector
from diffusers.utils import load_image

source    = load_image("dancer.png")
pose_map  = OpenposeDetector.from_pretrained("lllyasviel/Annotators")(source)
depth_map = MidasDetector.from_pretrained("lllyasviel/Annotators")(source)

pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    controlnet=[pose_cn, depth_cn], torch_dtype=torch.float16).to("cuda")

image = pipe(
    prompt="a dancer on a temple stage, dramatic light",
    image=[pose_map, depth_map],                       # one map per ControlNet
    controlnet_conditioning_scale=[0.8, 0.5],       # pose tight, depth loose
    num_inference_steps=28,
).images[0]
print("pose + depth composed")
# Output: pose + depth composed

# UNION MODEL: one ControlNet that accepts many condition types
# (InstantX FLUX.1-dev-Controlnet-Union). Convenient, but weaker at any one
# condition type than a specialized single-condition ControlNet - a tradeoff.

**Explanation**

This cell demonstrates that controls compose: you pass parallel *lists* of ControlNets, condition maps, and scales, so each control pins a different aspect at a different strength. The load-bearing detail is that every ControlNet still needs its own preprocessed map - never the raw photo - exactly as in the single-control cell.

**How the code works - step by step**
- **Load two ControlNets** - `pose_cn` (OpenPose) and `depth_cn` (depth), both in fp16.
- **Build one map per ControlNet** - `OpenposeDetector` makes `pose_map` and `MidasDetector` makes `depth_map` from the same `dancer.png` source.
- **`controlnet=[pose_cn, depth_cn]`** attaches both to the SDXL pipeline as a list.
- **`image=[pose_map, depth_map]`** passes one map per ControlNet, and **`controlnet_conditioning_scale=[0.8, 0.5]`** gives pose a tight scale and depth a loose one so they cooperate instead of fighting.
- **Trailing comment** notes the union-model alternative (one ControlNet accepting many condition types) - convenient but weaker per condition than a specialized model.

**In one line:** parallel lists of ControlNets, maps, and scales -> each control pins a different aspect.

**What you'll see:** Prints `pose + depth composed` - the generated (unsaved) image follows the dancer's pose tightly and the scene's depth loosely.

## 4 - Choosing the right control tool

Controlled generation is a toolbox, and picking the tool is most of the skill. This cell encodes the decision as a lookup - name the axis of control (structure, look, reusable style, editing, region) and it returns the right tool - plus per-model-family scale starting points, since an SDXL conditioning scale is too strong on Flux or SD3.5.

In [ ]:
# Match the tool to the axis of control, then start from a per-model scale.
def pick_control(goal: str) -> str:
    return {
        "pose or layout":          "ControlNet (structure)",
        "match one reference":       "IP-Adapter",
        "reusable brand style":      "LoRA (Lesson 6.2)",
        "edit this image":           "img2img / inpaint (Lesson 6.2)",
        "different content per area": "regional prompting",
    }[goal]

# conditioning_scale starting points differ by model family
SCALE = {"sdxl": 0.75, "flux": 0.5, "sd3.5": 0.5}

print(pick_control("pose or layout"), "| flux scale:", SCALE["flux"])
# Output: ControlNet (structure) | flux scale: 0.5

**Explanation**

A pure-Python reference cell, not a model call - a lookup table plus a scale dictionary. It captures two hard-won facts: match the tool to the axis of control, and start conditioning scales from per-model defaults rather than reusing one number everywhere.

**How the code works - step by step**
- **`pick_control(goal)`** - a dict lookup mapping each control axis (pose/layout, match one reference, reusable brand style, edit this image, different content per area) to its tool (ControlNet, IP-Adapter, LoRA, img2img/inpaint, regional prompting).
- **`SCALE`** - conditioning-scale starting points that differ by model family: SDXL 0.75, Flux 0.5, SD3.5 0.5.
- **`print(...)`** - looks up the tool for `pose or layout` and the Flux scale.

**In one line:** name the axis -> get the tool -> start from a per-model scale.

**What you'll see:** Prints `ControlNet (structure) | flux scale: 0.5` - no model runs; it is a decision reference.

## 5 - Control in production: reproducible and consent-gated

Shipping control means more than pixels: a run must be reproducible (pin every field) and responsible (block likeness generation without recorded consent). This cell captures a full run as a dict and runs a `consent_gate` before generation - an executable version of the ethical rule.

In [ ]:
import json

# A control run is fully described by these fields - pin ALL of them so a
# result is reproducible, not a lucky seed.
run = {
    "base": "stabilityai/stable-diffusion-xl-base-1.0",
    "controlnets": [("openpose", 0.8), ("depth", 0.5)],
    "ip_adapter": {"ref": "brand_style.jpg", "scale": 0.6},
    "seed": 42, "steps": 28,
}

def consent_gate(run, has_consent: bool) -> bool:
    """Block any likeness (pose / FaceID) without recorded consent."""
    # conservative: flag ANY IP-Adapter ref (it may be of a person) or a pose map;
    # in real use, narrow the IP-Adapter check to FaceID / person references.
    uses_identity = bool(run["ip_adapter"]) or \
        any("pose" in c for c, _ in run["controlnets"])
    return (not uses_identity) or has_consent

print("ship" if consent_gate(run, has_consent=True) else "blocked")
# Output: ship
print(json.dumps(run["controlnets"]))
# Output: [["openpose", 0.8], ["depth", 0.5]]

**Explanation**

This cell turns the production concerns into code: a `run` dict that fully describes a control pipeline, and a `consent_gate` function that runs *before* generation to block identity control (pose or any IP-Adapter reference) unless consent is recorded. It is a policy check plus a reproducibility record, not a model call.

**How the code works - step by step**
- **`run` dict** - pins the whole recipe: base model, ControlNets with scales, IP-Adapter ref and scale, seed, and steps, so the result is repeatable rather than seed-luck.
- **`consent_gate(run, has_consent)`** - flags `uses_identity` if the run has any IP-Adapter reference (conservatively, it could be a person) or a pose ControlNet, then returns True only if there is no identity use or consent is on record.
- **First `print`** - runs the gate with `has_consent=True`, so it ships.
- **Second `print`** - dumps the pinned ControlNet list as JSON, illustrating the reproducibility record.

**In one line:** pin the full run -> gate identity control on consent -> ship only if allowed.

**What you'll see:** Prints `ship` (the gate passes because consent is True) then `[["openpose", 0.8], ["depth", 0.5]]` - the pinned ControlNet config as JSON.

Across six cells you moved from "a prompt only weakly controls layout" to a full control stack: ControlNet pins structure from an extracted map, IP-Adapter transfers a reference's look with zero training, multi-ControlNet and regional prompting compose several controls at once, a decision table matches each tool to its axis of control, and a production run makes the whole recipe reproducible and consent-gated. The through-line is that control is a budget you spend where composition matters, tuned per model family. Next, Lesson 6.4 (ViT & CLIP) opens up the image encoder that makes IP-Adapter work in the first place.